# Final reference evaluation on Colab

Use a **T4 GPU**. The method and threshold are fixed from development. This notebook evaluates the 12 reviewed held-out extracts and compares routing with and without perplexity. It reviews the union of the two selections, generating each response once. No calibration or model training takes place on test data.

Enable the existing `HF_TOKEN` Secret. Keep it out of cells and Git. The optional last section runs the complete working corpus after the reference test has been inspected.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

repo = Path("/content/forensic-llm-graph")
if repo.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=repo, check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/AntoFratta/forensic-llm-graph.git", str(repo)], check=True)
os.chdir(repo)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)
print("Python:", sys.version.split()[0])


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)


In [ ]:
import json
from google.colab import userdata
from huggingface_hub import hf_hub_download

# The environment passes authentication to the child process without writing credentials.
token = userdata.get("HF_TOKEN")
if not token:
    raise RuntimeError("Add the HF_TOKEN Secret and enable notebook access.")
os.environ["HF_TOKEN"] = token
del token
settings = json.loads(Path("configs/hybrid.json").read_text(encoding="utf-8"))
hf_hub_download(settings["perplexity"]["model"], "config.json",
                revision=settings["perplexity"]["revision"])
print("Minerva access verified. No model weights loaded yet.")


## Upload the two input files

Upload these files to `/content/` using the Files panel:

- `05_batched_form_boundaries_spacy_lg.json`
- `test_reference.reviewed.json`, exported locally after both test review batches are confirmed

The reviewed reference remains outside Git. Labels are used only for metrics; the models receive the original text. No Drive mount or raw-data download is needed.


In [ ]:
input_path = Path("/content/05_batched_form_boundaries_spacy_lg.json")
reference_path = Path("/content/test_reference.reviewed.json")
for path in (input_path, reference_path):
    if not path.is_file():
        raise FileNotFoundError(f"Upload {path.name} using the Files panel before continuing.")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
preview_path = Path("outputs/10_final_test_preview.json")
subprocess.run([sys.executable, "scripts/run_hybrid.py", "--input", str(input_path),
                "--reference", str(reference_path), "--compare-routing", "--prepare-only",
                "--output", str(preview_path), "--overwrite"], check=True)
preview = json.loads(preview_path.read_text(encoding="utf-8"))
print("Scope:", preview["run"]["scope"])
print("Input documents:", preview["run"]["input_document_count"])
print("Full-corpus passages:", preview["run"]["full_passage_count"])
print("Test passages:", preview["run"]["passages_by_source"])
assert preview["run"]["scope"] == "reference_test"
assert preview["run"]["passages_by_source"] == {
    "redit": 3, "archivio_antimafia": 3, "dice": 3, "rizzo_synthetic": 3
}
del preview


## Run the fixed method

Minerva scores all 12 test windows and is released before Qwen loads. Each routing policy can select at most three of these windows under the fixed 20% budget. Qwen reviews their union: at most six distinct windows, fewer when selections overlap. The stored primary documents use the with-perplexity policy; comparison-only responses cannot alter that output.

The first Qwen download includes approximately 16.4 GB of source weights. Active runtimes reuse downloaded files. The GPU loading uses NF4 quantization. Progress and model-loading messages appear below.

If interrupted, append `--resume` to the same command. Existing final outputs are protected.


In [ ]:
def run_streaming(command):
    """Stream model progress while limiting long perplexity logs."""
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        try:
            for line in process.stdout:
                if line.startswith("Perplexity:"):
                    done, total = map(int, line.split(":", 1)[1].strip().split("/"))
                    if total > 100 and done % 100 and done != total:
                        continue
                print(line, end="", flush=True)
            return_code = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            process.wait()
            raise
        if return_code != 0:
            raise subprocess.CalledProcessError(return_code, command)

output_path = Path("outputs/10_final_test_models.json")
command = [sys.executable, "-u", "scripts/run_hybrid.py", "--input", str(input_path),
           "--reference", str(reference_path), "--compare-routing", "--output", str(output_path)]
# To continue an interrupted run: command.append("--resume")
run_streaming(command)


In [ ]:
evaluation_path = Path("outputs/10_final_test_evaluation.json")
subprocess.run([sys.executable, "scripts/evaluate.py", "--reference", str(reference_path),
                "--baseline", str(input_path), "--hybrid", str(output_path),
                "--output", str(evaluation_path), "--overwrite"], check=True)
results = json.loads(output_path.read_text(encoding="utf-8"))
run = results["run"]
print("Scope:", run["scope"])
print("Primary selected windows:", run["primary_selected_passage_count"])
print("Distinct model reviews including comparison:", run["reviewed_passage_count"])
print("Perplexity status:", run["perplexity_status_counts"])
print("LLM status:", run["review_status_counts"])
print("Generation attempts:", run["generation_attempt_count"])
print("GPU:", run["device"])
print("Elapsed seconds:", round(run["invocation_seconds"], 2))
print("Saved:", output_path, evaluation_path, sep="\n")


## Save the test results

Download `outputs/10_final_test_models.json` and `outputs/10_final_test_evaluation.json` manually for inspection. The results report exact spans and categories, with Rizzo synthetic diagnostics separated from REDIT forms, news and judicial texts. REDIT uses fictitious personal values. Inspect category support before interpreting F1.

The test is small and source-balanced; its investigative extracts come from one held-out PDF/case. It is not a representative estimate for all forensic language. Development did not identify a quality preference between the tried perplexity thresholds; the fixed threshold uses its 80th-percentile fallback. Test outcomes must not retune it.

If proceeding to the full-corpus section after inspection, keep this runtime so model downloads can be reused. Otherwise, save results and disconnect. Save a `.checkpoints.jsonl` manually before deleting an interrupted runtime when recovery is needed.


## Complete corpus execution after the reference test

This section is disabled by default. Enable it after inspecting the final test. Minerva scores every operational passage in the 10,651-document input. Qwen processes at most 100 prioritized windows, under the same frozen threshold and 20% fraction. The absolute cap limits GPU time; it is not a claim that only 100 windows are uncertain.

The full-corpus routing budget and distribution differ from the small reference experiment. The additional reference evaluation below scores the actual full-corpus output on the checked test cores, rather than extrapolating the small-subset result. It does not add a second global no-perplexity run.


In [ ]:
RUN_FULL_CORPUS = False

if RUN_FULL_CORPUS:
    full_path = Path("outputs/11_full_corpus_hybrid.json")
    full_command = [sys.executable, "-u", "scripts/run_hybrid.py", "--input", str(input_path),
                    "--output", str(full_path)]
    # To continue an interrupted full run: full_command.append("--resume")
    run_streaming(full_command)
    import hashlib
    from scripts.run_hybrid import code_hash
    from src.evaluation import load_reference, score_reference
    reference, reference_hash = load_reference(reference_path)
    full_result = json.loads(full_path.read_text(encoding="utf-8"))
    assert full_result["run"]["scope"] == "extended"
    assert full_result["run"]["input_sha256"] == hashlib.sha256(input_path.read_bytes()).hexdigest()
    assert full_result["run"]["code_sha256"] == code_hash()
    assert len(full_result["documents"]) == full_result["run"]["input_document_count"]
    full_metrics_path = Path("outputs/11_full_corpus_reference_evaluation.json")
    full_metrics = {"stage": "full_corpus_reference_evaluation", "reference_sha256": reference_hash,
                    "model_output_sha256": hashlib.sha256(full_path.read_bytes()).hexdigest(),
                    "run": full_result["run"], "routing": full_result["routing"],
                    "metrics": score_reference(reference, full_result["documents"])}
    full_metrics_path.write_text(json.dumps(full_metrics, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print("Full-corpus outputs:", full_path, full_metrics_path, sep="\n")
    print("Save both files manually before disconnecting.")
else:
    print("Full-corpus execution is disabled pending inspection of the final reference test.")
